# Notebook 03: Gold Layer — Weekly Demand Aggregation, Continuous Grid Construction & Feature Engineering

This notebook consumes the cleaned datasets from the **Silver Layer** (`data/2_silver/`), aggregates transactions into weekly SKU demand, constructs a continuous lifecycle calendar grid (zero-filling non-sales weeks), and generates time-series features (lags, rolling statistics, calendar seasonality, price trends) for machine learning forecasting.

---

##  1. Objectives & Architectural Decisions

* **Weekly Granularity Aggregation:** Truncate transaction timestamps (`InvoiceDate`) to weekly start boundaries (`Week_Start`, starting on Mondays) and aggregate total demand (`Weekly_Demand`), revenue (`Weekly_Revenue`), and average unit price (`Avg_UnitPrice`).
* **Continuous Grid Construction (Lifecycle-Constrained):** Generate a continuous 7-day weekly calendar grid spanning only between each SKU's `First_Week` and `Last_Week` of observed activity. This avoids artificial zero-filling prior to product launch or after product discontinuation.
* **Zero-Filling & Description Forward-Filling:** Fill missing weekly demand with `0`, missing revenue with `0.0`, and forward-fill missing product descriptions (`Description`) across weeks per SKU.
* **Time-Series Feature Engineering:**
  * **Target Definition:** `Target_Demand_Weekly` (Demand shifted by -1 week).
  * **Lag Features:** `Lag_1W`, `Lag_2W`, `Lag_4W`.
  * **Rolling Window Statistics (4-Week Window):** Shifted rolling mean (`Rolling_Mean_4W`) and rolling standard deviation (`Rolling_Std_4W`) to eliminate forward-looking data leakage.
  * **Demand Dynamics & Trend:** Demand trend ratio (`Demand_Trend_Ratio`) and coefficient of variation (`Demand_CV`).
  * **Price Ratio:** Current unit price relative to the 4-week historical moving average (`Price_Ratio_Vs_Avg`).
  * **Seasonal / Calendar Features:** `Month`, `Quarter`, `Year`, and peak season flag (`Is_Peak_Season` for November/December).

---

## 2. Execution Pipeline

In [36]:
# 1. Setup workspace directories
from pathlib import Path
import polars as pl
from loguru import logger

BASE_DIR = Path("..")
SILVER_DIR = BASE_DIR / "data" / "2_silver"
GOLD_DIR = BASE_DIR / "data" / "3_gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load Silver datasets
df_0910_clean = pl.read_parquet(SILVER_DIR / "sheet_2009_2010_cleaned.parquet")
df_1011_clean = pl.read_parquet(SILVER_DIR / "sheet_2010_2011_cleaned.parquet")

logger.info(f"Sheet 09-10 Silver: {len(df_0910_clean):,} dòng")
logger.info(f"Sheet 10-11 Silver: {len(df_1011_clean):,} dòng")

2026-11-06 14:05:41.946 | INFO     | __main__:<module>:15 - Sheet 09-10 Silver: 511,565 dòng
2026-11-06 14:05:41.949 | INFO     | __main__:<module>:16 - Sheet 10-11 Silver: 530,105 dòng


In [37]:

# 3. Concatenate sheets and truncate to weekly start
df_combined = pl.concat([df_0910_clean, df_1011_clean]).with_columns([
    pl.col("InvoiceDate").dt.truncate("1w").alias("Week_Start")
])
logger.info(f"Tổng dữ liệu sau khi gộp 2 sheet: {len(df_combined):,} dòng giao dịch.")

2026-11-06 14:05:42.003 | INFO     | __main__:<module>:5 - Tổng dữ liệu sau khi gộp 2 sheet: 1,041,670 dòng giao dịch.


Zero-Filling (Điền 0 cho những ngày sản phẩm không có giao dịch)

In [38]:

# 4. Weekly raw aggregation
df_weekly_raw = df_combined.group_by(["Week_Start", "StockCode"]).agg([
    pl.col("Quantity").sum().alias("Weekly_Demand"),
    (pl.col("Quantity") * pl.col("UnitPrice")).sum().alias("Weekly_Revenue"),
    pl.col("UnitPrice").mean().alias("Avg_UnitPrice"),
    pl.col("Description").mode().first().alias("Description")
]).sort(["StockCode", "Week_Start"])

In [39]:

# 5. Build continuous lifecycle calendar grid
start_week = df_weekly_raw["Week_Start"].min()
end_week = df_weekly_raw["Week_Start"].max()

calendar = pl.DataFrame({
    "Week_Start": pl.datetime_range(
        start=start_week,
        end=end_week,
        interval="1w",
        eager=True
    )
})

# Xác định tuần đầu và tuần cuối có giao dịch của từng SKU
sku_bounds = df_weekly_raw.group_by("StockCode").agg([
    pl.col("Week_Start").min().alias("First_Week"),
    pl.col("Week_Start").max().alias("Last_Week")
])

# Tạo Grid: chỉ bổ sung tuần nằm trong vòng đời quan sát của SKU
weekly_grid = (
    sku_bounds.join(calendar, how="cross")
    .filter(
        (pl.col("Week_Start") >= pl.col("First_Week")) &
        (pl.col("Week_Start") <= pl.col("Last_Week"))
    )
    .select(["Week_Start", "StockCode"])
)

# 6. Join aggregated sales onto grid and apply zero-filling
df_gold_weekly_full = (
    weekly_grid.join(
        df_weekly_raw,
        on=["Week_Start", "StockCode"],
        how="left"
    )
    .sort(["StockCode", "Week_Start"])
    .with_columns([
        pl.col("Weekly_Demand").fill_null(0),
        pl.col("Weekly_Revenue").fill_null(0.0),
        pl.col("Description").forward_fill().over("StockCode").fill_null("UNKNOWN")
    ])
)

logger.success(
    f"Weekly Grid hoàn tất: {len(df_gold_weekly_full):,} dòng"
)

2026-11-06 14:05:43.109 | SUCCESS  | __main__:<module>:45 - Weekly Grid hoàn tất: 324,607 dòng


In [40]:
# 7. Time-Series Feature Engineering & Memory Optimization
df_gold_final = (
    df_gold_weekly_full
    .sort(["StockCode", "Week_Start"])
    .with_columns([
        # Zero-Demand Flag (nghi vấn Out-of-Stock/Zero demand)
        (pl.col("Weekly_Demand") == 0).cast(pl.Int8).alias("Is_Zero_Demand"),

        # Target: nhu cầu và ngày khởi đầu của tuần kế tiếp
        pl.col("Weekly_Demand").shift(-1).over("StockCode").alias("Target_Demand_Weekly"),
        pl.col("Week_Start").shift(-1).over("StockCode").alias("Target_Week_Start"),

        # Lag Features
        pl.col("Weekly_Demand").shift(1).over("StockCode").alias("Lag_1W"),
        pl.col("Weekly_Demand").shift(2).over("StockCode").alias("Lag_2W"),
        pl.col("Weekly_Demand").shift(4).over("StockCode").alias("Lag_4W"),

        # Rolling Statistics (chỉ tính trên 4 tuần quá khứ)
        pl.col("Weekly_Demand").shift(1).rolling_mean(window_size=4).over("StockCode").alias("Rolling_Mean_4W"),
        pl.col("Weekly_Demand").shift(1).rolling_std(window_size=4).over("StockCode").alias("Rolling_Std_4W"),

        # Calendar Features (Downcast kiểu dữ liệu)
        pl.col("Week_Start").dt.month().cast(pl.Int8).alias("Month"),
        pl.col("Week_Start").dt.quarter().cast(pl.Int8).alias("Quarter"),
        pl.col("Week_Start").dt.year().cast(pl.Int16).alias("Year"),
        pl.col("Week_Start").dt.month().is_in([11, 12]).cast(pl.Int8).alias("Is_Peak_Season")
    ])
    .with_columns([
        # Trend & Volatility Ratios
        (pl.col("Lag_1W") / (pl.col("Rolling_Mean_4W") + 1.0)).alias("Demand_Trend_Ratio"),
        (pl.col("Rolling_Std_4W") / (pl.col("Rolling_Mean_4W") + 1.0)).alias("Demand_CV"),

        # Price Ratio: giá hiện tại so với giá trung bình 4 tuần trước
        (
            pl.col("Avg_UnitPrice") /
            (pl.col("Avg_UnitPrice").shift(1).rolling_mean(window_size=4).over("StockCode") + 1e-5)
        ).alias("Price_Ratio_Vs_Avg")
    ])
    .filter(
        pl.col("Lag_4W").is_not_null() &
        pl.col("Target_Demand_Weekly").is_not_null()
    )
)

logger.success(
    f"Feature Engineering hoàn tất: {df_gold_final.height:,} dòng, "
    f"{len(df_gold_final.columns)} cột"
)

2026-11-06 14:05:43.395 | SUCCESS  | __main__:<module>:45 - Feature Engineering hoàn tất: 301,631 dòng, 21 cột


In [41]:
# 8. Export final Gold dataset to Parquet
path_gold = GOLD_DIR / "weekly_demand_features_calendar_fixed.parquet"
df_gold_final.write_parquet(path_gold)

logger.success(f"Đã lưu Gold mới: {path_gold}")

2026-11-06 14:05:43.644 | SUCCESS  | __main__:<module>:5 - Đã lưu Gold mới: ../data/3_gold/weekly_demand_features_calendar_fixed.parquet


3. Data Integrity & Validation Audits

In [42]:
# 1. Null Checks on Critical Features
critical_cols = [
    "Weekly_Demand", "Target_Demand_Weekly", 
    "Lag_1W", "Lag_2W", "Lag_4W", 
    "Rolling_Mean_4W", "Rolling_Std_4W"
]

display(
    df_gold_final.select([
        pl.col(col).null_count().alias(col)
        for col in critical_cols
    ])
)

# 2. Duplicate SKU-Week Check
duplicates = (
    df_gold_final
    .group_by(["StockCode", "Week_Start"])
    .len()
    .filter(pl.col("len") > 1)
)

# 3. Weekly Continuity (7-Day Gap) Audit
gap_check = (
    df_gold_weekly_full
    .sort(["StockCode", "Week_Start"])
    .with_columns(
        pl.col("Week_Start").diff().over("StockCode").dt.total_days().alias("Gap_Days")
    )
    .filter(
        pl.col("Gap_Days").is_not_null() &
        (pl.col("Gap_Days") != 7)
    )
)

print(f"Duplicate SKU-Weeks: {duplicates.height}")
print(f"Abnormal Weekly Gaps (!= 7 Days): {gap_check.height}")

Weekly_Demand,Target_Demand_Weekly,Lag_1W,Lag_2W,Lag_4W,Rolling_Mean_4W,Rolling_Std_4W
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0


Duplicate SKU-Weeks: 0
Abnormal Weekly Gaps (!= 7 Days): 0
